# 01 — Ingest and harmonize the raw PCOS dataset

Reads `data_raw/pcos_data_base_ml.xlsx`, restricts to the PCOS group, coalesces the multiple raw column-name variants per biomarker into the 35 predefined candidate features (endocrine/metabolic/inflammatory/thyroid + AGE/BMI/CRP), computes the deterministic derived indices used only in the secondary sensitivity network, and applies the predefined inclusion rule (missingness ≤ 45%, > 5 unique values) to determine the primary and secondary biomarker sets (Methods 2.4, 2.5).

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT / "outputs").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if _in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import numpy as np, pandas as pd

raw = pd.read_excel(PROJECT_ROOT / 'data_raw' / 'pcos_data_base_ml.xlsx')
df = raw.copy()
if 'group' in df.columns:
    df = df[df['group'].astype(str).str.upper().eq('PCOS')].copy()

H = pd.DataFrame({'row_id': np.arange(len(df))})
maps = []
for f, c in CAND.items():
    H[f], used = coalesce(df, c)
    maps.append({
        'feature': f, 'columns_used': ' | '.join(used),
        'n_nonmissing': int(H[f].notna().sum()), 'missing_pct': float(H[f].isna().mean() * 100),
    })

print(f'Participants after PCOS-group filter: {len(H)}')


### Derived clinical indices (secondary sensitivity network only)

Deterministic mathematical transformations of primary measurements (Methods 2.4) — excluded from the primary network to avoid mathematical coupling.

In [ ]:
H['LH_FSH_ratio'] = H['LH'] / H['FSH']
H['FAI'] = H['TESTO_TOTAL'] / H['SHBG'] * 100
H['HOMA_IR'] = H['GLU0'] * H['INS0'] / 405
H['QUICKI'] = 1 / (np.log10(H['INS0']) + np.log10(H['GLU0']))
H['TyG'] = np.log((H['TG'] * H['GLU0']) / 2)
H['TG_HDL_ratio'] = H['TG'] / H['HDL']
H['non_HDL'] = H['TCHOL'] - H['HDL']
H['CORT_ratio_23_8'] = H['CORT_23'] / H['CORT_8']
H['NLR'] = H['NEUT_ABS'] / H['LYMPH_ABS']
H['PLR'] = H['PLT'] / H['LYMPH_ABS']
H['MLR'] = H['MONO_ABS'] / H['LYMPH_ABS']
H['SII'] = H['PLT'] * H['NEUT_ABS'] / H['LYMPH_ABS']
H = H.replace([np.inf, -np.inf], np.nan)


### Primary / secondary feature-set determination and save outputs

In [ ]:
all_dom = {**DOM, **DERIVED}
primary = [f for f in DOM if f in H.columns and H[f].isna().mean() <= 0.45 and H[f].nunique(dropna=True) > 5]
secondary = primary + [f for f in DERIVED if f in H.columns and H[f].isna().mean() <= 0.45 and H[f].nunique(dropna=True) > 5]

fd = pd.DataFrame([
    {
        'feature': f, 'domain': all_dom.get(f, 'clinical'),
        'feature_type': 'derived_sensitivity_only' if f in DERIVED else ('direct_primary_candidate' if f in DOM else 'clinical'),
        'included_primary': f in primary,
        'included_secondary_full_clinical': f in secondary,
        'n_nonmissing': int(H[f].notna().sum()), 'missing_pct': float(H[f].isna().mean() * 100),
    }
    for f in H.columns if f != 'row_id'
])

out_dir = PROJECT_ROOT / 'outputs' / '01_ingest_harmonize'
out_dir.mkdir(parents=True, exist_ok=True)
H.to_csv(out_dir / 'harmonized_features.csv', index=False)
pd.DataFrame(maps).to_csv(out_dir / 'feature_mapping_report.csv', index=False)
fd.to_csv(out_dir / 'feature_dictionary_v2.csv', index=False)

print(f'Primary direct biomarkers: {len(primary)}')
print(f'Secondary (incl. derived) biomarkers: {len(secondary)}')
fd.head(10)
